In [8]:
pip install fastexcel

  Using cached fastexcel-0.19.0-cp310-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached fastexcel-0.19.0-cp310-abi3-win_amd64.whl (2.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install polars openpyxl


SyntaxError: invalid syntax (2659163020.py, line 1)

In [2]:
import polars as pl
from pathlib import Path
import fastexcel

In [3]:
def read_excel(path: Path, sheet_pattern: str | None = None, had_header: bool = True) -> pl.DataFrame:
    """
    Read matching sheets from an Excel file into a single Polars DataFrame.
    
    - sheet_pattern: prefix to match sheet names (e.g. "Income" matches "Income - 1", "Income - 2").
                     None = read all sheets.
    - had_header: whether the sheet has a header row.
    """
    reader = fastexcel.read_excel(path)
    print(f"  Sheets in {path.name}: {reader.sheet_names}")

    # Filter sheets by pattern
    sheets = [s for s in reader.sheet_names
              if sheet_pattern is None or s.startswith(sheet_pattern)]
    print(f"  Matching sheets: {sheets}")

    dfs = []
    for name in sheets:
        df = pl.read_excel(path, sheet_name=name, has_header=had_header)
        df = df.with_columns(pl.lit(name).alias("_source_sheet"))
        dfs.append(df)

    if not dfs:
        return pl.DataFrame()

    combined = pl.concat(dfs, how="diagonal_relaxed") if len(dfs) > 1 else dfs[0]
    return combined.with_columns(pl.lit(path.name).alias("_source_file"))

In [4]:
def concat_files(files: list[Path], label: str, sheet_pattern: str | int | None = None, has_header: bool = True) -> pl.DataFrame:
    if not files:
        return pl.DataFrame()
    dfs = []
    for f in files:
        df = read_excel(f, sheet_pattern=sheet_pattern, had_header=has_header)
        if label == "income_all" and not has_header:
            header = df.row(2)
            df = df.slice(3)
            df.columns = header
            dfs.append(df)
        elif label == "balance_all" and not has_header:
            header =df.row(13)
            df = df.slice(14)
            df.columns = header
            dfs.append(df)
        print(f"  Loaded {f.name}: {df.shape}")
    combined = pl.concat(dfs, how="diagonal_relaxed")
    # before = combined.height
    # Drop duplicates (exclude _source_file so rows from overlapping files are caught)
    # data_cols = [c for c in combined.columns if c != "_source_file"]
    # combined = combined.unique(subset=data_cols, keep="first")
    # after = combined.height
    # dupes = before - after
    # print(f"=> {label}: {combined.shape} (removed {dupes} duplicate rows)\n")
    return combined

In [5]:
root = Path(r"c:\Users\TanJunJie\OneDrive - SRKK Group\Project\watson_entriesmatching\OneDrive_2026-03-09\Shopee Sample Reports (Testing)\scenario1")

# Find all .xlsx files recursively
xlsx_files = sorted(root.rglob("*.xlsx"))

## Income Report


In [6]:
# --- Load & concatenate by report type ---
income_files = [f for f in xlsx_files if f.name.startswith("Income.released")]
print("=== Income Released ===")
income_all = concat_files(income_files, "income_all", sheet_pattern="Income", has_header=False)



=== Income Released ===
  Sheets in Income.released.my.20251231_20251231.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Income - 1', 'Adjustment', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20251231_20251231.xlsx: (15021, 49)
  Sheets in Income.released.my.20260101_20260101.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260101_20260101.xlsx: (12599, 49)
  Sheets in Income.released.my.20260102_20260102.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']
  Matching sheets: ['Income - 1', 'Income - 2']
  Loaded Income.released.my.20260102_20260102.xlsx: (16012, 49)
  Sheets in Income.released.my.20260103_20260103.xlsx: ['Summary', 'Service Fee Details', 'Shipping Fee Discrepancy', 'Adjustment', 'Income - 1', 'Income - 2']


In [46]:
income_all.head()

Sequence No.,View By,Order ID,Shop Name,refund id,Product ID,Product Name,Order Creation Date,Payout Completed Date,Release Channel,Order Type,Hot Listing,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Amount Paid By Buyer,Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Shipping Fee Promotion by Seller,Shipping provider,Courier Name,Voucher Code From Seller,Lost Compensation,Cash refund to buyer amount,Pro-rated coin offset for return refund Items,Pro-rated Shopee voucher offset for returned items,Pro-rated Bank Payment Channel Promotion for return refund Items,Pro-rated Shopee Payment Channel Promotion for return refund Items,Income - 1,Income.released.my.20251231_20251231.xlsx,Income.released.my.20260101_20260101.xlsx,Income.released.my.20260102_20260102.xlsx,Income.released.my.20260103_20260103.xlsx,Income.released.my.20260104_20260104.xlsx,Income.released.my.20260105_20260105.xlsx,Income.released.my.20260106_20260106.xlsx,Income.released.my.20260107_20260107.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1""","""Order""","""251230QHF7YGBE""","""Watsons Malaysia""","""""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-5.19""","""27.93""","""-27.93""","""0""","""-4.9""","""-0.29""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""ndkshidshisbssayacantik""","""27.93""","""3.78""","""Cash on Delivery""","""""","""""","""0.00""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""0.00""","""-27.93""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""2""","""Sku""","""251230QHF7YGBE""","""Watsons Malaysia""","""-""","""28850540679""","""DOVE Body Scrub Pomegranate 28…","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-5.19""","""0""","""-27.93""","""0""","""-4.9""","""-0.29""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""-""","""""","""-""","""-""","""Cash on Delivery""","""""","""""","""-""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""-""","""-""","""-""","""-""","""-""","""-""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""3""","""Order""","""251230PMP6MTW6""","""Watsons Malaysia""","""220867667213152""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-2.65""","""14.9""","""-14.9""","""0""","""-2.5""","""-0.15""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""gq6pe2qreq""","""14.90""","""3.78""","""Online Banking""","""""","""""","""0.00""","""Self Collection Point""","""Self Collection Point (SPX Exp…","""""","""0.00""","""-14.90""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""4""","""Sku""","""251230PMP6MTW6""","""Watsons Malaysia""","""-""","""16492683485""","""Orita Charcoal Dehumidifier (3…","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-2.65""","""0""","""-14.9""","""0""","""-2

In [47]:
print(f'Duplicate rows: {income_all.is_duplicated().sum()}')

Duplicate rows: 0


In [53]:
# Filter to only "Order" view (exclude "Settlement" which has overlapping data but different granularity)
income_all = income_all.filter(pl.col("View By")== "Order")
income_all.head()

Sequence No.,View By,Order ID,Shop Name,refund id,Product ID,Product Name,Order Creation Date,Payout Completed Date,Release Channel,Order Type,Hot Listing,Total Released Amount (RM),Product Price,Refund Amount,Shipping Fee Paid by Buyer (excl. SST),Shipping Fee Charged by Logistic Provider,Seller Paid Shipping Fee SST,Shipping Rebate From Shopee,Reverse Shipping Fee,Reverse Shipping Fee SST,Saver Programme Shipping Fee Savings,Return to Seller Fee,Rebate Provided by Shopee,Voucher Sponsored by Seller,Coin Cashback Sponsored by Seller,Commission Fee (incl. SST),Service Fee (Incl. SST),Transaction Fee (Incl. SST),AMS Commission Fee,Saver Programme Fee (Incl. SST),Username (Buyer),Amount Paid By Buyer,Transaction Fee Rate (%),Buyer Payment Method,Buyer Payment Method Details_1(if applicable),Payment Details / Installment Plan,Shipping Fee Promotion by Seller,Shipping provider,Courier Name,Voucher Code From Seller,Lost Compensation,Cash refund to buyer amount,Pro-rated coin offset for return refund Items,Pro-rated Shopee voucher offset for returned items,Pro-rated Bank Payment Channel Promotion for return refund Items,Pro-rated Shopee Payment Channel Promotion for return refund Items,Income - 1,Income.released.my.20251231_20251231.xlsx,Income.released.my.20260101_20260101.xlsx,Income.released.my.20260102_20260102.xlsx,Income.released.my.20260103_20260103.xlsx,Income.released.my.20260104_20260104.xlsx,Income.released.my.20260105_20260105.xlsx,Income.released.my.20260106_20260106.xlsx,Income.released.my.20260107_20260107.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""1""","""Order""","""251230QHF7YGBE""","""Watsons Malaysia""","""""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-5.19""","""27.93""","""-27.93""","""0""","""-4.9""","""-0.29""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""ndkshidshisbssayacantik""","""27.93""","""3.78""","""Cash on Delivery""","""""","""""","""0.00""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""0.00""","""-27.93""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""3""","""Order""","""251230PMP6MTW6""","""Watsons Malaysia""","""220867667213152""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""-2.65""","""14.9""","""-14.9""","""0""","""-2.5""","""-0.15""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0.00""","""gq6pe2qreq""","""14.90""","""3.78""","""Online Banking""","""""","""""","""0.00""","""Self Collection Point""","""Self Collection Point (SPX Exp…","""""","""0.00""","""-14.90""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""5""","""Order""","""251230PB5YG8HT""","""Watsons Malaysia""","""""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""37.32""","""44.1""","""0""","""0""","""-4.9""","""0""","""4.9""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""-4.64""","""0""","""-2.14""","""0""","""0.00""","""fiqamokhtar""","""40.38""","""4.86""","""SPayLater""","""""","""12x""","""0.00""","""Doorstep Delivery""","""SPX Express (West Malaysia)""","""""","""0.00""","""0.00""","""0.00""","""0.00""","""0.00""","""0.00""","""Income - 1""","""Income.released.my.20251231_20…",null,null,null,null,null,null,null
"""7""","""Order""","""251230PB3GD63E""","""Watsons Malaysia""","""""","""-""","""-""","""2025-12-30""","""2025-12-31""","""Seller Wallet""","""Normal Order""","""NO""","""34.51""","""41.04""","""0""","""0""","""-4.9""","""0""","""4.9""","""0""",

## Balance Report

In [7]:
balance_files = [f for f in xlsx_files if f.name.startswith("my_balance_transaction")]
print("=== Balance Transaction ===")
balance_all = concat_files(balance_files, "balance_all", sheet_pattern="Transaction", has_header=False)

=== Balance Transaction ===
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx: (5000, 10)
  Sheets in my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: ['Transaction Report']
  Matching sheets: ['Transaction Report']
  Loaded my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx: (5000, 10)
  Sheets

In [78]:
balance_all.head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""2026-01-07 23:59:53""","""Order Income""","""Income from Order #251227FMQA3…","""251227FMQA3XGP""","""Money In""","""13.42""","""Transaction Completed""","""195391.02""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null
"""2026-01-07 23:59:46""","""Order Income""","""Income from Order #260102VN9VM…","""260102VN9VMVPV""","""Money In""","""8.31""","""Transaction Completed""","""195377.6""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null
"""2026-01-07 23:59:32""","""Order Income""","""Income from Order #2601045P6TK…","""2601045P6TK6PG""","""Money In""","""21.25""","""Transaction Completed""","""195369.29""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null
"""2026-01-07 23:59:28""","""Order Income""","""Income from Order #2601045Y4YT…","""2601045Y4YTEG6""","""Money In""","""13.19""","""Transaction Completed""","""195348.04""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null
"""2026-01-07 23:59:26""","""Order Income""","""Income from Order #2601045E6UG…","""2601045E6UGTNU""","""Money In""","""23.08""","""Transaction Completed""","""195334.85""","""Transaction Report""","""my_balance_transaction_report.…",null,null,null,null,null,null,null,null


In [79]:
balance_all.is_duplicated().sum()

0

In [86]:
balance_all = balance_all.with_columns(
    pl.col("Date").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

In [81]:
balance_all.select(pl.col("Transaction Type").unique())

Transaction Type
str
"""Order Income"""
"""Withdrawals"""
"""Adjustment"""


In [ ]:
balance_all.select(pl.col("Status").unique())

: 

In [90]:
balance_all.select(
    pl.col("Date").min()
)

Date
datetime[μs]
2025-12-31 00:01:02


In [91]:
balance_all.select(
    pl.col("Date").max()
)

Date
datetime[μs]
2026-01-07 23:59:53


In [88]:
balance_all_withdrawal = balance_all.filter(pl.col("Transaction Type") == "Withdrawals")
balance_all_withdrawal.head()

Date,Transaction Type,Description,Order ID,Money Direction,Amount,Status,Balance After Transactions,Transaction Report,my_balance_transaction_report.shopee.20251231_20260107part_1_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_2_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_3_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_4_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_5_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_6_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_7_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_8_of_9.xlsx,my_balance_transaction_report.shopee.20251231_20260107part_9_of_9.xlsx
datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-9388.4""","""Transaction Completed""","""0""","""Transaction Report""",null,"""my_balance_transaction_report.…",null,null,null,null,null,null,null
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-150000""","""Transaction Completed""","""9388.4""","""Transaction Report""",null,"""my_balance_transaction_report.…",null,null,null,null,null,null,null
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-150000""","""Transaction Completed""","""159388.4""","""Transaction Report""",null,"""my_balance_transaction_report.…",null,null,null,null,null,null,null
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-150000""","""Transaction Completed""","""309388.4""","""Transaction Report""",null,"""my_balance_transaction_report.…",null,null,null,null,null,null,null
2026-01-07 02:32:18,"""Withdrawals""","""Automated Withdrawal""","""-""","""Money Out""","""-150000""","""Transaction Completed""","""459388.4""","""Transaction Report""",null,"""my_balance_transaction_report.…",null,null,null,null,null,null,null


## Sales Report

In [8]:
sales_files = sorted(root.parent.glob("SalesReport*.xlsx"))
print("=== Sales Reports ===")
print([f.name for f in sales_files])

sales_dfs = []
for f in sales_files:
    df = pl.read_excel(f, sheet_name="SalesReport", has_header=True)
    df = df.with_columns(pl.lit(f.name).alias("_source_file"))
    sales_dfs.append(df)

sales_report = pl.concat(sales_dfs, how="diagonal_relaxed") if sales_dfs else pl.DataFrame()
print("sales_report shape:", sales_report.shape)

sales_report.head()

=== Sales Reports ===
['SalesReport Wk2.xlsx', 'SalesReport Wk3.xlsx']


Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 8, falling back to string


sales_report shape: (62427, 10)


OrderNum,MarketPlaceOrderNum,SalesGlobalTxnID,eStoreCode,SalesWorkDate,TotalAmount,PayId,TenderType,TlogCount,_source_file
str,str,i64,i64,date,f64,str,str,str,str
"""9524949786""","""2601044PUM2781""",233603239,895,2026-01-05,21.0,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524950020""","""26010450MFPQE1""",233603241,895,2026-01-05,98.29,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524951072""","""2601045R7NTP28""",233603245,895,2026-01-05,107.65,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524928466""","""260101TUY6HNTV""",233603250,895,2026-01-05,9.56,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""
"""9524930857""","""260101UF58WU38""",233603253,895,2026-01-05,2.73,null,"""eMarket Place""",null,"""SalesReport Wk2.xlsx"""


# Reconciliation

In [124]:
print('income_all columns:', income_all.columns)
print('balance_all columns:', balance_all.columns)
print('sales_report columns:', sales_report.columns)

print('\nIncome sample Order ID:')
print(income_all.select(pl.col('Order ID')).head(10))

print('\nBalance sample Description:')
print(balance_all.select(pl.col('Description')).head(10))

print('\nSales sample Marketplace Order ID:')
print(sales_report.select(pl.col('MarketPlaceOrderNum')).head(10))

income_all columns: ['Sequence No.', 'View By', 'Order ID', 'Shop Name', 'refund id', 'Product ID', 'Product Name', 'Order Creation Date', 'Payout Completed Date', 'Release Channel', 'Order Type', 'Hot Listing', 'Total Released Amount (RM)', 'Product Price', 'Refund Amount', 'Shipping Fee Paid by Buyer (excl. SST)', 'Shipping Fee Charged by Logistic Provider', 'Seller Paid Shipping Fee SST', 'Shipping Rebate From Shopee', 'Reverse Shipping Fee', 'Reverse Shipping Fee SST', 'Saver Programme Shipping Fee Savings', 'Return to Seller Fee', 'Rebate Provided by Shopee', 'Voucher Sponsored by Seller', 'Coin Cashback Sponsored by Seller', 'Commission Fee (incl. SST)', 'Service Fee (Incl. SST)', 'Transaction Fee (Incl. SST)', 'AMS Commission Fee', 'Saver Programme Fee (Incl. SST)', 'Username (Buyer)', 'Amount Paid By Buyer', 'Transaction Fee Rate (%)', 'Buyer Payment Method', 'Buyer Payment Method Details_1(if applicable)', 'Payment Details / Installment Plan', 'Shipping Fee Promotion by Seller

In [125]:
# Canonicalize key fields and numeric/date types for reconciliation
income_norm = (
    income_all
    .select([
        'Order ID',
        'View By',
        'Order Creation Date',
        'Payout Completed Date',
        'Total Released Amount (RM)',
    ])
    .filter(pl.col('View By') == 'Order')
    .with_columns([
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars().alias('order_id'),
        pl.col('Total Released Amount (RM)').cast(pl.Float64, strict=False).alias('released_amount'),
        pl.col('Order Creation Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False).alias('order_creation_date'),
        pl.col('Payout Completed Date').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False).alias('payout_completed_date'),
    ])
    .drop_nulls(['order_id'])
)

balance_norm = (
    balance_all
    .select([
        'Date',
        'Transaction Type',
        'Description',
        'Order ID',
        'Money Direction',
        'Amount',
        'Status',
    ])
    .with_columns([
        pl.col('Date').cast(pl.Utf8).str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False).alias('txn_datetime'),
        pl.col('Amount').cast(pl.Float64, strict=False).alias('amount'),
        pl.col('Transaction Type').cast(pl.Utf8).str.strip_chars().alias('transaction_type'),
        pl.col('Order ID').cast(pl.Utf8).str.strip_chars().alias('order_id_from_col'),
        pl.col('Description').cast(pl.Utf8).str.extract(r'#([A-Z0-9]+)', group_index=1).str.strip_chars().alias('order_id_from_desc'),
    ])
    .with_columns([
        pl.coalesce([pl.col('order_id_from_col'), pl.col('order_id_from_desc')]).alias('order_id'),
    ])
)

sales_norm = (
    sales_report
    .select([
        'OrderNum',
        'MarketPlaceOrderNum',
        'SalesWorkDate',
        'TotalAmount',
    ])
    .with_columns([
        pl.col('MarketPlaceOrderNum').cast(pl.Utf8).str.strip_chars().alias('order_id'),
        pl.col('TotalAmount').cast(pl.Float64, strict=False).alias('sales_gross_amount'),
        pl.coalesce([
            pl.col('SalesWorkDate').cast(pl.Date, strict=False),
            pl.col('SalesWorkDate').cast(pl.Utf8).str.strptime(pl.Date, '%d-%m-%y', strict=False),
            pl.col('SalesWorkDate').cast(pl.Utf8).str.strptime(pl.Date, '%Y-%m-%d', strict=False),
        ]).alias('sales_work_date'),
    ])
    .drop_nulls(['order_id'])
)

print('income_norm:', income_norm.shape)
print('balance_norm:', balance_norm.shape)
print('sales_norm:', sales_norm.shape)

income_norm: (41871, 9)
balance_norm: (41904, 13)
sales_norm: (62427, 7)


In [126]:
income_norm.head()

Order ID,View By,Order Creation Date,Payout Completed Date,Total Released Amount (RM),order_id,released_amount,order_creation_date,payout_completed_date
str,str,str,str,str,str,f64,date,date
"""251230QHF7YGBE""","""Order""","""2025-12-30""","""2025-12-31""","""-5.19""","""251230QHF7YGBE""",-5.19,2025-12-30,2025-12-31
"""251230PMP6MTW6""","""Order""","""2025-12-30""","""2025-12-31""","""-2.65""","""251230PMP6MTW6""",-2.65,2025-12-30,2025-12-31
"""251230PB5YG8HT""","""Order""","""2025-12-30""","""2025-12-31""","""37.32""","""251230PB5YG8HT""",37.32,2025-12-30,2025-12-31
"""251230PB3GD63E""","""Order""","""2025-12-30""","""2025-12-31""","""34.51""","""251230PB3GD63E""",34.51,2025-12-30,2025-12-31
"""251230PAY9XQWY""","""Order""","""2025-12-30""","""2025-12-31""","""26.79""","""251230PAY9XQWY""",26.79,2025-12-30,2025-12-31


In [127]:
# Core reconciliation table: one row per income order_id
recon_core = (
    income_by_order
    .join(balance_income_by_order, on='order_id', how='left')
    .join(sales_by_order, on='order_id', how='left')
    .with_columns([
        pl.col('balance_order_income_amount').fill_null(0.0),
        pl.col('sales_total_amount').fill_null(0.0),
    ])
    .with_columns([
        (pl.col('income_released_amount') - pl.col('balance_order_income_amount')).alias('diff_income_vs_balance'),
        (pl.col('sales_total_amount') - pl.col('income_released_amount')).alias('diff_sales_vs_income'),
    ])
    .with_columns([
        pl.when(pl.col('sales_total_amount') == 0)
            .then(pl.lit('Missing in Sales'))
            .when(pl.col('diff_income_vs_balance').abs() <= 0.01)
            .then(pl.lit('Income-Balance Matched'))
            .otherwise(pl.lit('Income-Balance Mismatch'))
            .alias('recon_status'),
    ])
)

recon_summary = recon_core.select([
    pl.len().alias('income_orders'),
    (pl.col('diff_income_vs_balance').abs() <= 0.01).sum().alias('income_balance_matched_orders'),
    (pl.col('sales_total_amount') == 0).sum().alias('missing_in_sales_orders'),
    pl.col('income_released_amount').sum().alias('income_total'),
    pl.col('balance_order_income_amount').sum().alias('balance_income_total'),
    pl.col('sales_total_amount').sum().alias('sales_total'),
    pl.col('diff_income_vs_balance').sum().alias('total_diff_income_vs_balance'),
])

recon_summary

income_orders,income_balance_matched_orders,missing_in_sales_orders,income_total,balance_income_total,sales_total,total_diff_income_vs_balance
u32,u32,u32,f64,f64,f64,f64
41871,41871,33995,1.5643e6,1.5643e6,350097.29,0.0


In [128]:
# Exception tables for investigation
missing_in_income_from_balance = balance_income_by_order.join(income_by_order, on='order_id', how='anti')
missing_in_sales_from_income = income_by_order.join(sales_by_order, on='order_id', how='anti')

income_balance_mismatch = (
    recon_core
    .filter(pl.col('diff_income_vs_balance').abs() > 0.01)
    .select([
        'order_id',
        'income_released_amount',
        'balance_order_income_amount',
        'diff_income_vs_balance',
        'income_order_date',
        'income_payout_date',
        'balance_first_txn',
        'balance_last_txn',
        'income_row_count',
        'balance_income_row_count',
    ])
    .sort(pl.col('diff_income_vs_balance').abs(), descending=True)
)

print('missing_in_income_from_balance:', missing_in_income_from_balance.shape)
print('missing_in_sales_from_income:', missing_in_sales_from_income.shape)
print('income_balance_mismatch:', income_balance_mismatch.shape)

income_balance_mismatch.head(20)

missing_in_income_from_balance: (0, 5)
missing_in_sales_from_income: (33995, 5)
income_balance_mismatch: (0, 10)


order_id,income_released_amount,balance_order_income_amount,diff_income_vs_balance,income_order_date,income_payout_date,balance_first_txn,balance_last_txn,income_row_count,balance_income_row_count
str,f64,f64,f64,date,date,datetime[μs],datetime[μs],u32,u32


In [129]:
# Sales key diagnostics: overlap between Income order_id and Sales MarketPlaceOrderNum
sales_overlap = income_by_order.join(sales_by_order, on='order_id', how='inner')

print('income order ids:', income_by_order.height)
print('sales order ids:', sales_by_order.height)
print('overlap order ids:', sales_overlap.height)

print('\nSample income orders missing in sales:')
print(missing_in_sales_from_income.select('order_id').head(20))

print('\nSample sales marketplace order ids:')
print(sales_by_order.select('order_id').head(20))

income order ids: 41871
sales order ids: 62427
overlap order ids: 7876

Sample income orders missing in sales:
shape: (20, 1)
┌────────────────┐
│ order_id       │
│ ---            │
│ str            │
╞════════════════╡
│ 251220TPB794SY │
│ 251227FC41XFF5 │
│ 251218P48KKV71 │
│ 251214BS43594V │
│ 251228H7SU9QNW │
│ …              │
│ 2512114MWHFVFV │
│ 2512126UQ2V5PG │
│ 251212740AJSRF │
│ 251230PGCXX6TE │
│ 260101T79F4YET │
└────────────────┘

Sample sales marketplace order ids:
shape: (20, 1)
┌────────────────┐
│ order_id       │
│ ---            │
│ str            │
╞════════════════╡
│ 2601033S4JD8A1 │
│ 2601142J87KBFA │
│ 26010593T7MB2M │
│ 260111QXPNSMCR │
│ 260106BGEX13QN │
│ …              │
│ 2601069JCNHJ7G │
│ 260106AK8N9NNV │
│ 2601059BDR1B0S │
│ 26011548X1N3KM │
│ 2601155BBN7UDU │
└────────────────┘


In [130]:
# Date-window diagnostics
print('income order date range:')
print(income_norm.select([
    pl.col('order_creation_date').min().alias('min_order_date'),
    pl.col('order_creation_date').max().alias('max_order_date'),
    pl.col('payout_completed_date').min().alias('min_payout_date'),
    pl.col('payout_completed_date').max().alias('max_payout_date'),
]))

print('\nsales work date range:')
print(sales_norm.select([
    pl.col('sales_work_date').min().alias('min_sales_work_date'),
    pl.col('sales_work_date').max().alias('max_sales_work_date'),
]))

income_min = income_norm.select(pl.col('payout_completed_date').min()).item()
income_max = income_norm.select(pl.col('payout_completed_date').max()).item()
sales_min = sales_norm.select(pl.col('sales_work_date').min()).item()
sales_max = sales_norm.select(pl.col('sales_work_date').max()).item()

print('\nincome payout range:', income_min, 'to', income_max)
print('sales work date range:', sales_min, 'to', sales_max)

income order date range:
shape: (1, 4)
┌────────────────┬────────────────┬─────────────────┬─────────────────┐
│ min_order_date ┆ max_order_date ┆ min_payout_date ┆ max_payout_date │
│ ---            ┆ ---            ┆ ---             ┆ ---             │
│ date           ┆ date           ┆ date            ┆ date            │
╞════════════════╪════════════════╪═════════════════╪═════════════════╡
│ 2025-11-21     ┆ 2026-01-06     ┆ 2025-12-31      ┆ 2026-01-07      │
└────────────────┴────────────────┴─────────────────┴─────────────────┘

sales work date range:
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_sales_work_date ┆ max_sales_work_date │
│ ---                 ┆ ---                 │
│ date                ┆ date                │
╞═════════════════════╪═════════════════════╡
│ 2026-01-05          ┆ 2026-01-18          │
└─────────────────────┴─────────────────────┘

income payout range: 2025-12-31 to 2026-01-07
sales work date range: 2026-01-05 to 2026-01-18


In [131]:
# Aggregate each report to order-level amounts
income_by_order = (
    income_norm
    .group_by("order_id")
    .agg([
        pl.col("released_amount").sum().alias("income_released_amount"),
        pl.col("order_creation_date").min().alias("income_order_date"),
        pl.col("payout_completed_date").max().alias("income_payout_date"),
        pl.len().alias("income_row_count"),
    ])
)

balance_income_by_order = (
    balance_norm
    .filter(pl.col("transaction_type") == "Order Income")
    .group_by("order_id")
    .agg([
        pl.col("amount").sum().alias("balance_order_income_amount"),
        pl.col("txn_datetime").min().alias("balance_first_txn"),
        pl.col("txn_datetime").max().alias("balance_last_txn"),
        pl.len().alias("balance_income_row_count"),
    ])
)

sales_by_order = (
    sales_norm
    .group_by("order_id")
    .agg([
        pl.col("sales_gross_amount").sum().alias("sales_total_amount"),
        pl.col("sales_work_date").min().alias("sales_work_date"),
        pl.len().alias("sales_row_count"),
    ])
)

print("income_by_order:", income_by_order.shape)
print("balance_income_by_order:", balance_income_by_order.shape)
print("sales_by_order:", sales_by_order.shape)

income_by_order: (41871, 5)
balance_income_by_order: (41827, 5)
sales_by_order: (62427, 4)


## Reconciliation Blueprint (Order-level)

We will reconcile in 2 layers:
1. **Income vs Balance**: `Income.released[Order ID]` against Balance `Description -> #OrderID` for `Transaction Type = Order Income`.
2. **Income vs Sales**: `Income.released[Order ID]` against `SalesReport[MarketPlaceOrderNum]`.

`SalesReport[TotalAmount]` is generally a gross sales value, while `Income.released[Total Released Amount (RM)]` is net released value, so differences are expected and should be explained by fee/refund columns.